In [3]:
import torch
import vllm

print("vLLM:", vllm.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

vLLM: 0.31.0
PyTorch: 2.13.0+cu130
CUDA available: True
GPU: Tesla T4


In [3]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)

Python executable: /usr/bin/python3
Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [1]:
import torch
import vllm

print("vLLM:", vllm.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

vLLM: 0.31.0
PyTorch: 2.13.0+cu130
CUDA available: True
GPU: Tesla T4


--------------------------------------------------------------------------------

vLLM loads the model onto the available GPU, starts an API server, and waits for clients to send prompts. It then generates responses and returns them through the API.

In [12]:
!vllm serve Qwen/Qwen2.5-0.5B-Instruct \
    --host 0.0.0.0 \
    --port 8001 \
    --api-key day1-test-key \
    --dtype half \
    --max-model-len 2048 \
    --gpu-memory-utilization 0.70

(APIServer pid=1853) INFO 10-16 16:41:39 [api_utils.py:347] 
(APIServer pid=1853) INFO 10-16 16:41:39 [api_utils.py:347]        █     █     █▄   ▄█
(APIServer pid=1853) INFO 10-16 16:41:39 [api_utils.py:347]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.31.0
(APIServer pid=1853) INFO 10-16 16:41:39 [api_utils.py:347]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-0.5B-Instruct
(APIServer pid=1853) INFO 10-16 16:41:39 [api_utils.py:347]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=1853) INFO 10-16 16:41:39 [api_utils.py:347] 
(APIServer pid=1853) INFO 10-16 16:41:39 [api_utils.py:286] non-default args: {'model_tag': 'Qwen/Qwen2.5-0.5B-Instruct', 'host': '0.0.0.0', 'port': 8001, 'api_key': '***', 'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'dtype': 'half', 'max_model_len': 2048, 'gpu_memory_utilization': 0.7}
(APIServer pid=1853) INFO 10-16 16:41:39 [model.py:699] Resolved architecture: Qwen2ForCausalLM
(APIServer pid=1853) WARNING 10-16 16:41:39 [model.py:2442] Casting torch.bfloat16 to torch.float16

In [11]:
!pkill -f VLLM::EngineCore  #to clear GPU memory

In [3]:
!nvidia-smi

Fri Oct 16 16:43:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Start the server 

In [2]:
import subprocess
import time
import requests

with open("/content/vllm-server.log", "w") as log:
    process = subprocess.Popen(
        [
            "vllm", "serve", "Qwen/Qwen2.5-0.5B-Instruct",
            "--host", "0.0.0.0",
            "--port", "8001",
            "--api-key", "day1-test-key"
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

print("Server PID:", process.pid)

Server PID: 2008


### check the server is runnning or not

In [4]:
import time
import requests

for _ in range(90):
    try:
        response = requests.get(
            "http://127.0.0.1:8001/health",
            timeout=2
        )
        if response.status_code == 200:
            print("vLLM server is ready!")
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print("Server did not become ready.")



vLLM server is ready!


In [8]:
import requests

response = requests.post(
    "http://localhost:8001/v1/chat/completions",
    headers={"Authorization": "Bearer day1-test-key"},
    json={
        "model": "Qwen/Qwen2.5-0.5B-Instruct",
        "messages": [
            {
                "role": "user",
                "content": "who is narendra modi"
            }
        ],
        "max_tokens": 50
    },
    timeout=120
)

print("Status:", response.status_code)
print(response.json()["choices"][0]["message"]["content"])

Status: 200
Narendra Modi is the current Prime Minister of India and the leader of the Bharatiya Janata Party (BJP). Born on January 23, 1950, in Allahabad, Uttar Pradesh, Modi has been instrumental in
